# E29 --- o outro teto: os bits

O corte do capítulo 3 é contado com um número por dia, e são seis mil setecentos e dezoito deles.
Guardar cem números no lugar de seis mil e setecentos é o gesto mais natural do mundo, e ele tem
preço tabelado. Esta medição cobra três coisas do resumo: **o erro** que ele comete ao estimar a
energia do caminho, **o controle** num mundo que não muda, e **o que ele não responde** --- porque a
pergunta do livro é sobre a ordem dos dias, e o resumo não guarda ordem nenhuma.


In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, regimes, resumo, volatilidade

RAIZ = Path.cwd()
CONTADORES = resumo.CONTADORES_PADRAO
MUNDOS = resumo.MUNDOS_PADRAO
PERMUTACOES = 40
SEMENTE = 20260924
CAUDA = 0.05

serie = dados.carregar_serie("sp500.csv")
retornos = volatilidade.retornos_log(serie).to_numpy()
rng = np.random.default_rng(SEMENTE)
print("frevolab %s | %d dias | contadores %s" % (frevolab.VERSAO, retornos.size, CONTADORES))
print("energia exata do caminho: %.6f" % resumo.energia(retornos))


frevolab 0.1.0 | 6718 dias | contadores (25, 100, 400, 1600, 10000)
energia exata do caminho: 0.988366


In [2]:
# Painel 1 --- a lei do preco: o erro contra o numero de contadores.
sorteio = np.random.default_rng(SEMENTE + 1)
linhas = []
for k in CONTADORES:
    r = resumo.varredura(retornos, k, MUNDOS, sorteio)
    linhas.append({"contadores": k, "erro_medio": r["erro_medio"], "previsto": r["previsto"],
                   "razao": r["erro_medio"] / r["previsto"], "erro_p95": r["erro_p95"]})
quadro = pd.DataFrame(linhas)
print(quadro.to_string(index=False, float_format=lambda v: "%.4f" % v))
ajuste = np.polyfit(np.log([l["contadores"] for l in linhas]), np.log([l["erro_medio"] for l in linhas]), 1)
print()
print("inclinacao medida no log-log: %.3f (a lei preve -0,5)" % ajuste[0])
print("cem contadores medem %.1f%% | dez mil medem %.1f%%"
      % (100 * linhas[1]["erro_medio"], 100 * linhas[-1]["erro_medio"]))
print("e o preco tabelado: um por cento custa %d contadores" % resumo.contadores_para(0.01))


 contadores  erro_medio  previsto  razao  erro_p95
         25      0.2289    0.2828 0.8092    0.5064
        100      0.1221    0.1414 0.8631    0.3374
        400      0.0589    0.0707 0.8329    0.1334
       1600      0.0312    0.0354 0.8834    0.0755
      10000      0.0110    0.0141 0.7771    0.0234

inclinacao medida no log-log: -0.505 (a lei preve -0,5)
cem contadores medem 12.2% | dez mil medem 1.1%
e o preco tabelado: um por cento custa 20000 contadores


In [3]:
# Figura 1: o erro medido contra o numero de contadores, com a reta da lei ao lado.
fig, eixo = plt.subplots(figsize=(7.6, 4.0))
ks = np.array([l["contadores"] for l in linhas], dtype=float)
eixo.loglog(ks, [l["erro_medio"] for l in linhas], marker="o", lw=1.8, color="#1f4e79",
            label="erro medido (média de %d sorteios)" % MUNDOS)
eixo.loglog(ks, [l["previsto"] for l in linhas], lw=1.4, ls="--", color="#b03a2e",
            label="a lei: raiz de 2/K")
eixo.set_xlabel("contadores guardados")
eixo.set_ylabel("erro relativo na energia")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, which="both", ls=":")
fig.tight_layout()
graficos.salvar(fig, "E29_bits", 1)
plt.close(fig)
print("figura gravada")


figura gravada


In [4]:
# Painel 2 --- o controle (um mundo que nao muda) e a moeda da decisao.
constante = np.ones(retornos.size)
sorteio = np.random.default_rng(SEMENTE + 2)
controle = {k: resumo.varredura(constante, k, MUNDOS, sorteio) for k in (100, 10000)}
print("mundo constante, energia exata %.1f (sabida por construcao: n)" % resumo.energia(constante))
for k, r in controle.items():
    print("  %5d contadores: erro medio %.4f | previsto %.4f" % (k, r["erro_medio"], r["previsto"]))

# A decisao: o alarme precisa de um corte, e o resumo entrega uma energia. A ponte entre os
# dois supoe simetria, e a taxa assinada e o que essa suposicao cobra.
exato = resumo.energia(retornos)
corte_limiar = float(np.quantile(retornos, CAUDA))
corte_gaussiano = resumo.corte_normal(exato, retornos.size, CAUDA)
esboco = resumo.esboco(retornos, 100, np.random.default_rng(SEMENTE + 3))
corte_do_resumo = resumo.corte_normal(esboco["estimativa"], retornos.size, CAUDA)
taxas = {"limiar do dado": float((retornos < corte_limiar).mean()),
         "gaussiana da energia exata": float((retornos < corte_gaussiano).mean()),
         "gaussiana do esboco de cem": float((retornos < corte_do_resumo).mean())}
for nome, taxa in taxas.items():
    print("%-28s assina %.3f%%" % (nome, 100 * taxa))
print("promessa declarada: %.1f%%" % (100 * CAUDA))


mundo constante, energia exata 6718.0 (sabida por construcao: n)
    100 contadores: erro medio 0.1159 | previsto 0.1414
  10000 contadores: erro medio 0.0106 | previsto 0.0141
limiar do dado               assina 5.001%
gaussiana da energia exata   assina 4.302%
gaussiana do esboco de cem   assina 3.498%
promessa declarada: 5.0%


In [5]:
# Painel 3 --- o que o resumo nao responde: a ordem.
sorteio = np.random.default_rng(SEMENTE + 4)
piores, energias, erros = [], [], []
for _ in range(PERMUTACOES):
    embaralhado = sorteio.permutation(retornos)
    energias.append(resumo.energia(embaralhado))
    niveis = 100.0 * np.exp(np.concatenate([[0.0], np.cumsum(embaralhado)]))
    piores.append(regimes.estatisticas(niveis[1:])["pior"])
    erros.append(resumo.esboco(embaralhado, 100, sorteio)["erro"])
piores = np.array(piores)
print("energia exata: original %.6f | maior desvio entre %d embaralhamentos %.2e"
      % (exato, PERMUTACOES, np.max(np.abs(np.array(energias) - exato)) / exato))
print("pior bloco de sessenta dias: %d no dado | de %d a %d entre os embaralhamentos"
      % (regimes.estatisticas(serie.to_numpy()[1:])["pior"], piores.min(), piores.max()))
print("erro do esboco de cem nos embaralhados: media %.4f (sem embaralhar, %.4f)"
      % (np.mean(erros), linhas[1]["erro_medio"]))


energia exata: original 0.988366 | maior desvio entre 40 embaralhamentos 4.49e-16
pior bloco de sessenta dias: 56 no dado | de 42 a 60 entre os embaralhamentos
erro do esboco de cem nos embaralhados: media 0.1000 (sem embaralhar, 0.1221)


## Leitura visual das figuras

Feita nesta sessão abrindo o @@E29_bits_1.png@@ com a ponte de visão (AGENTS.md §9), depois de o
caderno rodar, e conferida contra o @@E29_bits.json@@.

**O que o desenho mostra.** Duas linhas em escalas duplas nos dois eixos: a medida (cheia, com
marcadores) e a lei (tracejada). A tracejada fica **acima** da medida em toda a extensão do eixo, e
as duas são **paralelas** --- a distância entre elas não abre nem fecha.

**E os números concordam.** A razão medida sobre previsto vai de 0,79 a 0,89 nos cinco pontos, e o
valor teórico dessa razão --- a média de uma estimativa simétrica contra o próprio desvio, que é a
raiz de dois sobre pi --- é 0,798. Não é só a inclinação que a lei acerta: é também a constante. A
figura é a mesma coisa que a tabela, desenhada.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
por_k = {l["contadores"]: l for l in linhas}
resultado = {
    "bits_dias": int(retornos.size),
    "bits_energia": round(exato, 6),
    "bits_contadores_menor": int(min(CONTADORES)),
    "bits_contadores_maior": int(max(CONTADORES)),
    "bits_mundos": int(MUNDOS),
    "bits_semente": int(SEMENTE),
    "bits_inclinacao": round(float(ajuste[0]), 3),
    "bits_contadores_para_um_por_cento": int(resumo.contadores_para(0.01)),
    "bits_contadores_para_dez_por_cento": int(resumo.contadores_para(0.10)),
}
NOMES = {25: "vinte_e_cinco", 100: "cem", 400: "quatrocentos", 1600: "mil_e_seiscentos",
         10000: "dez_mil"}
for k, nome in NOMES.items():
    resultado["bits_erro_%s" % nome] = round(100 * por_k[k]["erro_medio"], 1)
    resultado["bits_%s" % nome] = int(por_k[k]["contadores"])
    resultado["bits_previsto_%s" % nome] = round(100 * por_k[k]["previsto"], 1)
    if k in controle:
        resultado["bits_constante_%s" % nome] = round(100 * controle[k]["erro_medio"], 1)
resultado["bits_taxa_limiar"] = round(100 * taxas["limiar do dado"], 2)
resultado["bits_taxa_gaussiana"] = round(100 * taxas["gaussiana da energia exata"], 2)
resultado["bits_taxa_esboco"] = round(100 * taxas["gaussiana do esboco de cem"], 2)
resultado["bits_permutacoes"] = int(PERMUTACOES)
resultado["bits_desvio_energia"] = float(np.max(np.abs(np.array(energias) - exato)) / exato)
resultado["bits_pior_dado"] = int(regimes.estatisticas(serie.to_numpy()[1:])["pior"])
resultado["bits_pior_menor"] = int(piores.min())
resultado["bits_pior_maior"] = int(piores.max())
resultado["bits_erro_embaralhado"] = round(100 * float(np.mean(erros)), 1)
caminho = Path("lab/resultados/E29_bits.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E29_bits.json gravado | 35 grandezas
